# Quang Huy Shop — Colab Service

Notebook này chạy **AI microservice** tạm thời trên Google Colab (GPU) và expose qua **ngrok**.

Backend chính (FastAPI) gọi endpoint `POST /tryon` với prompt do hệ thống sinh từ thuộc tính sản phẩm.

## Cách dùng
1. **Runtime → Change runtime type → T4 GPU**
2. Chạy lần lượt các cell (hoặc *Run all*)
3. Copy URL ngrok in ra ở cell cuối → dán vào `backend/.env`:
   ```
   AI_SERVICE_URL=https://xxxx.ngrok-free.app
   ```
4. Mỗi lần mở Colab mới, URL ngrok đổi → cập nhật lại env và restart backend.

**Lưu ý:** Session Colab ~12h; inference ~30–90 giây/ảnh.

In [ ]:
# Cell 1 — Cài thư viện & tải SAM checkpoint
!pip install -q segment-anything diffusers transformers accelerate controlnet_aux safetensors opencv-python-headless fastapi uvicorn nest_asyncio python-multipart pyngrok

import os
SAM_CHECKPOINT = "sam_vit_b_01ec64.pth"
if not os.path.exists(SAM_CHECKPOINT):
    !wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

print("OK — dependencies & SAM checkpoint ready")

In [ ]:
# Cell 2 — Load models (SAM + Depth + SD ControlNet Inpaint)
import torch
from segment_anything import sam_model_registry, SamPredictor
from diffusers import StableDiffusionControlNetInpaintPipeline, ControlNetModel, UniPCMultistepScheduler
from transformers import pipeline as hf_pipeline

SAM_CHECKPOINT = "sam_vit_b_01ec64.pth"
MODEL_TYPE = "vit_b"
DEPTH_MODEL = "Intel/dpt-large"
CONTROLNET_ID = "lllyasviel/control_v11f1p_sd15_depth"
INPAINT_ID = "Uminosachi/realisticVisionV51_v51VAE-inpainting"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"Device: {device}")

print("Loading SAM vit_b...")
sam = sam_model_registry[MODEL_TYPE](checkpoint=SAM_CHECKPOINT)
sam.to(device=device)
sam_predictor = SamPredictor(sam)

print("Loading depth estimator...")
depth_estimator = hf_pipeline("depth-estimation", model=DEPTH_MODEL, device=0 if device == "cuda" else -1)

print("Loading ControlNet + Inpaint pipeline...")
controlnet = ControlNetModel.from_pretrained(CONTROLNET_ID, torch_dtype=dtype)
pipe = StableDiffusionControlNetInpaintPipeline.from_pretrained(
    INPAINT_ID,
    controlnet=controlnet,
    torch_dtype=dtype,
    safety_checker=None,
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
if device == "cuda":
    pipe.enable_model_cpu_offload()
else:
    pipe.to(device)

print("Models loaded.")

In [ ]:
# Cell 3 — Pipeline core (khớp backend/app/ai/config.py)
import cv2
import numpy as np
from PIL import Image, ImageFilter

# Mặc định trùng backend AI_INFERENCE — backend có thể override qua form
DEFAULT_INFERENCE = {
    "process_res": 768,
    "expansion": 50,
    "control_scale": 0.35,
    "num_inference_steps": 30,
    "guidance_scale": 7.5,
}


def get_depth_map(image):
    if isinstance(image, np.ndarray):
        image = Image.fromarray(image)
    depth_result = depth_estimator(image)["depth"]
    depth_arr = np.array(depth_result)
    depth_arr = depth_arr[:, :, None]
    depth_arr = np.concatenate([depth_arr, depth_arr, depth_arr], axis=2)
    return Image.fromarray(depth_arr)


def dilate_mask(mask_array, expansion_amount):
    if expansion_amount <= 0:
        return mask_array
    kernel_size = int(expansion_amount)
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    return cv2.dilate(mask_array, kernel, iterations=1)


def run_curtain_tryon_core(
    img_pil,
    prompt,
    neg_prompt,
    bbox,
    expansion,
    control_scale,
    process_res,
    num_inference_steps=DEFAULT_INFERENCE["num_inference_steps"],
    guidance_scale=DEFAULT_INFERENCE["guidance_scale"],
):
    image_pil = img_pil.convert("RGB")
    w_orig, h_orig = image_pil.size
    process_size = int(process_res)
    process_size = max(512, min(1024, process_size))

    image_resized = image_pil.resize((process_size, process_size), resample=Image.LANCZOS)

    if bbox is None:
        bbox_arr = np.array([0, 0, w_orig, h_orig])
    else:
        x_min, y_min, x_max, y_max = bbox
        x_min = max(0, min(w_orig - 1, x_min))
        x_max = max(1, min(w_orig, x_max))
        y_min = max(0, min(h_orig - 1, y_min))
        y_max = max(1, min(h_orig, y_max))
        bbox_arr = np.array([x_min, y_min, x_max, y_max])

    sam_predictor.set_image(np.array(image_pil))
    masks, _, _ = sam_predictor.predict(
        point_coords=None,
        point_labels=None,
        box=bbox_arr[None, :],
        multimask_output=False,
    )
    sam_mask_uint8 = (masks[0] * 255).astype(np.uint8)

    mask_pil_temp = Image.fromarray(sam_mask_uint8).resize(
        (process_size, process_size), resample=Image.NEAREST
    )
    mask_np_temp = np.array(mask_pil_temp)

    scaled_expansion = int(expansion * (process_size / 512))
    mask_dilated_np = dilate_mask(mask_np_temp, scaled_expansion)
    mask_pil = Image.fromarray(mask_dilated_np).convert("L")
    mask_pil = mask_pil.filter(ImageFilter.GaussianBlur(radius=5))

    blurred_image = image_resized.filter(ImageFilter.GaussianBlur(radius=30))
    image_for_inpaint_pil = Image.composite(blurred_image, image_resized, mask_pil)

    control_image = get_depth_map(image_resized)

    if device == "cuda":
        torch.cuda.empty_cache()

    output = pipe(
        prompt=prompt,
        negative_prompt=neg_prompt,
        image=image_for_inpaint_pil,
        mask_image=mask_pil,
        control_image=control_image,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        controlnet_conditioning_scale=float(control_scale),
    ).images[0]

    return output.resize((w_orig, h_orig), resample=Image.LANCZOS)


print("Pipeline core ready.")

In [ ]:
# Cell 4 — FastAPI + ngrok (backend gọi POST /tryon)
import threading
from io import BytesIO

import nest_asyncio
import uvicorn
from fastapi import FastAPI, File, Form, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, StreamingResponse
from PIL import Image
from pyngrok import ngrok

nest_asyncio.apply()

app = FastAPI(title="Quang Huy Shop — Colab Service", version="1.0")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/health")
def health():
    return {"status": "ok", "device": device, "inference_defaults": DEFAULT_INFERENCE}


@app.post("/tryon")
async def tryon_curtain(
    image: UploadFile = File(...),
    prompt: str = Form(...),
    neg_prompt: str = Form(""),
    x_min: int = Form(0),
    y_min: int = Form(0),
    x_max: int = Form(0),
    y_max: int = Form(0),
    expansion: int = Form(DEFAULT_INFERENCE["expansion"]),
    control_scale: float = Form(DEFAULT_INFERENCE["control_scale"]),
    process_res: int = Form(DEFAULT_INFERENCE["process_res"]),
):
    if not prompt.strip():
        return JSONResponse({"error": "prompt is required"}, status_code=400)

    try:
        img_bytes = await image.read()
        img_pil = Image.open(BytesIO(img_bytes)).convert("RGB")
    except Exception as exc:
        return JSONResponse({"error": f"invalid image: {exc}"}, status_code=400)

    bbox = None
    if x_max > x_min and y_max > y_min:
        bbox = (x_min, y_min, x_max, y_max)

    final_neg = neg_prompt.strip() or (
        "window frame distortion, messy, artifacts, bad anatomy, low quality, "
        "blurry, broken wall, ugly, text, watermark, cartoon, illustration"
    )

    try:
        result_pil = run_curtain_tryon_core(
            img_pil,
            prompt.strip(),
            final_neg,
            bbox,
            int(expansion),
            float(control_scale),
            int(process_res),
        )
    except Exception as exc:
        return JSONResponse({"error": str(exc)}, status_code=500)

    buf = BytesIO()
    result_pil.save(buf, format="PNG")
    buf.seek(0)
    return StreamingResponse(buf, media_type="image/png")


def _run_uvicorn():
    import asyncio

    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())


# Dừng tunnel cũ (nếu chạy lại cell)
try:
    ngrok.kill()
except Exception:
    pass

server_thread = threading.Thread(target=_run_uvicorn, daemon=True)
server_thread.start()

public_tunnel = ngrok.connect(8000)
public_url = public_tunnel.public_url

print("=" * 60)
print("Service local : http://127.0.0.1:8000")
print("Health check  : http://127.0.0.1:8000/health")
print("Docs          : http://127.0.0.1:8000/docs")
print("-" * 60)
print("NGROK URL (copy vào backend AI_SERVICE_URL):")
print(public_url)
print("=" * 60)
print("Test nhanh: GET", public_url + "/health")

## Kiểm tra thủ công (tuỳ chọn)

Sau khi backend cấu hình `AI_SERVICE_URL`, thử try-on từ frontend:
1. Admin cấu hình thuộc tính sản phẩm (Màu sắc, Chất liệu, …)
2. Trang chi tiết SP → **Thử rèm AI** → upload ảnh → vẽ khung cửa → **Tạo preview**

Cell dưới upload ảnh test trực tiếp lên Colab (không qua backend).

In [ ]:
# Cell 5 (tuỳ chọn) — Test trực tiếp trên Colab
# from google.colab import files
# uploaded = files.upload()
# test_path = next(iter(uploaded))
# img = Image.open(test_path).convert("RGB")
# w, h = img.size
# bbox = (int(w * 0.2), int(h * 0.1), int(w * 0.8), int(h * 0.85))
# prompt = "A pair of beige solid curtains, linen, blackout, soft natural folds, ceiling to floor, highly detailed fabric texture, interior design photography, realistic lighting, 8k, sharp focus"
# neg = "window frame distortion, messy, artifacts, low quality, blurry, text, watermark"
# out = run_curtain_tryon_core(img, prompt, neg, bbox, 50, 0.35, 768)
# display(out)